<div style="font-family: 'Times New Roman', serif;">

**Problem: Optimal Dice Stopping Game**
You are playing a game with a fair, six-sided die, which you may roll up to three times.
After seeing the roll, you have two choices:
1) Stop: Accept a payoff equal to the face value of the current roll (in dollars) and terminate the game.
2) Continue: Forfeit the current payoff completely and roll again. 
Assuming you play with an optimal strategy to maximize your expected payoff, what is the expected value of this game?

**Approach for Optimal stopping : Backword Induction / Dynamic Programming**

</div>



In [5]:
import math

dice = [1, 2, 3, 4, 5, 6]
T = 5

# payoff at last step
payoff = sum(dice) / len(dice)


for i in range(T-1, 0, -1):  # T-1 down to 1

    print(f"===Stage [{i}]===]")
    print(f"Continue if value <= {round(payoff, 2)}")

    # decision : Stop if x >= payoff (payoff : value of the dice), else continue (payoff : expected payoff of i+1)
    payoff = sum(max(x, payoff) for x in dice) / len(dice)
    
    print(f"Expected payoff : {round(payoff, 2)}\n")

print(payoff)

===Stage [4]===]
Continue if value <= 3.5
Expected payoff : 4.25

===Stage [3]===]
Continue if value <= 4.25
Expected payoff : 4.67

===Stage [2]===]
Continue if value <= 4.67
Expected payoff : 4.94

===Stage [1]===]
Continue if value <= 4.94
Expected payoff : 5.13

5.12962962962963


<div style="font-family: 'Times New Roman', serif;">

**What if the probabilities are different for each sides of the dice?**

</div>

In [6]:

# different prob for each side
dice_outcomes = [
    (1, 0.1),
    (2, 0.1),
    (3, 0.2),
    (4, 0.2),
    (5, 0.1),
    (6, 0.3)
]

T = 5

payoff = sum(v*p for v,p in dice_outcomes)

for stage in range(T-1, 0, -1):  # T-1 down to 1

    print(f"===Stage [{stage}]===]")
    print(f"Continue if value < {round(payoff, 2)}")


    payoff = sum(max(x, payoff) * p for x, p in dice_outcomes) 
    print(f"Expected payoff : {round(payoff, 2)}\n")


===Stage [4]===]
Continue if value < 4.0
Expected payoff : 4.7

===Stage [3]===]
Continue if value < 4.7
Expected payoff : 5.12

===Stage [2]===]
Continue if value < 5.12
Expected payoff : 5.38

===Stage [1]===]
Continue if value < 5.38
Expected payoff : 5.57



<div style="font-family: 'Times New Roman', serif;">

**What if there are costs for 1 more try? What if the value of the payout is discounted?**

</div>

In [7]:
# different prob for each side
dice_outcomes = [
    (1, 0.1),
    (2, 0.1),
    (3, 0.2),
    (4, 0.2),
    (5, 0.1),
    (6, 0.3)
]

T = 5
cost = 0.5
gamma = 0.9

payoff = sum(v*p for v,p in dice_outcomes)

for stage in range(T-1, 0, -1):  # T-1 down to 1

    print(f"===Stage [{stage}]===]")
    print(f"Continue if value < {round(payoff, 2)}")

    # decision : Stop if x >= payoff-cost, else continue 
    payoff = sum(max(x, payoff * gamma - cost) * p for x, p in dice_outcomes) 
    print(f"Expected payoff : {round(payoff, 2)}\n")

===Stage [4]===]
Continue if value < 4.0
Expected payoff : 4.34

===Stage [3]===]
Continue if value < 4.34
Expected payoff : 4.46

===Stage [2]===]
Continue if value < 4.46
Expected payoff : 4.51

===Stage [1]===]
Continue if value < 4.51
Expected payoff : 4.52



<div style="font-family: 'Times New Roman', serif;">

**What if the value of the payout is discounted BUT the trys are unlimited?**

$$V_t = \mathbb{E}[max(X, \gamma V_{t+1} - c)]$$
becomes

$$V = \mathbb{E}[max(X, \gamma V - c)]$$

Think of "stage" as # of turns left.  
1) When no turn is left
2) 1 turn is left
3) 2 turn is left  
...

</div>

In [10]:
# different prob for each side
dice_outcomes = [
    (1, 0.1),
    (2, 0.1),
    (3, 0.2),
    (4, 0.2),
    (5, 0.1),
    (6, 0.3)
]

cost = 0.5
gamma = 0.9

tolerance = 1e-7

i = 0
payoff_before = sum(v*p for v,p in dice_outcomes)
while True: 

    payoff_after = sum(max(x, payoff_before * gamma - cost) * p for x, p in dice_outcomes) 

    if abs(payoff_after - payoff_before) < tolerance:
        break

    payoff_before = payoff_after

    print(f"expected payoff[{i}]: {payoff_after}")
    i += 1

print("\n")
print(f"converged expected payoff: {payoff_after}")

expected payoff[0]: 4.34
expected payoff[1]: 4.4624
expected payoff[2]: 4.506464
expected payoff[3]: 4.52232704
expected payoff[4]: 4.5280377344
expected payoff[5]: 4.530093584384
expected payoff[6]: 4.53083369037824
expected payoff[7]: 4.531100128536167
expected payoff[8]: 4.5311960462730205
expected payoff[9]: 4.531230576658287
expected payoff[10]: 4.531243007596983
expected payoff[11]: 4.531247482734914
expected payoff[12]: 4.531249093784569
expected payoff[13]: 4.531249673762445
expected payoff[14]: 4.53124988255448


converged expected payoff: 4.531249957719613


<div style="font-family: 'Times New Roman', serif;">

### Proof that unique point of convergence exists

**Banach Fixed-Point Theorem**  

Let $(\mathcal{X}, d)$ be a non-empty complete metric space.  
If an operator $T: \mathcal{X} \to \mathcal{X}$ is a **contraction mapping** - meaning there exists a constant $\kappa \in [0,1)$ such that for all $u,v\in \mathcal{X}$:
$$d(T(u), T(v))\leq \kappa d(u,v)$$

$$\text{or... } \frac{|T(u)-T(v)|}{|u-v|} \leq \kappa$$
$\to$ function T shrinks(contracts) the distance between any two points by at least a factor of $\kappa$. This means $|T'(x)|\leq \kappa < 1$


Then :
1. There exists a unique fixed point $V^*\in \mathcal{X}$ such that $T(V^*) = V^*$  
$\to$ function T must meet with y = x at one single point
2. For any arbitrary initial point $V_0 \in \mathcal{X}$, the iterative sequence $V_{k+1} = T(V_k)$ converges 
$$\lim_{k\to\infty}V_k = V^*$$
3. The error decays exponentially:
$$d(V_k, V^*) \leq \frac{\kappa^k}{1-\kappa}d(V_1, V_0)$$


</br></br>


**Bellman Equation**

Define an operator $T$ (Bellman operator) that maps a candidate value function $V$ to an updated value function $TV$.  

The operator $T: \mathbb{R} \to \mathbb{R}$ is defined as: 
$$T(V) = \mathbb{E}[max(X, \gamma V - c)]$$

</br></br>

**Is the above bellman operator a contraction?**  
Take two values $u, v\in \mathbb{R}$
$$|T(u)-T(v)| = |\mathbb{E}[max(X, \gamma u - c)] - \mathbb{E}[max(X, \gamma v - c)]|$$
$$= |\mathbb{E}[max(X, \gamma u - c) - max(X, \gamma v - c)]| $$
By the Jensen Inequality, $\mathbb{E}[g(X)] \geq g(E[X])$ when $g$ is a convex function. Hence
$$|\mathbb{E}[max(X, \gamma u - c) - max(X, \gamma v - c)]| \leq \mathbb{E}[|max(X, \gamma u - c) - max(X, \gamma v - c)|]$$

Since $|max(a,b) - max(a,c)|\leq |b-c|$ 
$$|\max(X, \gamma u - c) - max(X, \gamma v - c)| \leq \gamma|u-v|$$

As a result,
$$|T(u)-T(v)| \leq \mathbb{E}[|max(X, \gamma u - c) - max(X, \gamma v - c)|] = \gamma|u-v|$$

$\therefore$ There exists an unique $V$ s.t. $T(V) = \mathbb{E}[max(X, \gamma V - c)] = V$



</div>